# Lab 2: Bash Options, Backups, and a Shell Game
Complete the three questions below. Each question states the task and expected output.

- Use Bash and the scripting tools covered in class. Complete the numbered TODOs in each starter script. Keep the supplied structure and helper functions.
- Replace each `todo NUMBER` call with your Bash code. The supplied `todo` helper stops unfinished code with a reminder; it is not part of your solution logic.
- Run each script cell before its test cells. Keep your code and test output in the notebook.
- Play Question 3 in a terminal using `bash aceydeucy.sh help`. Notebook test cells use piped input instead of live keyboard input.

## Requirements
1. You are expected to complete **at least 1 problem during the 107-minute lab session**.
2. **Before you leave**, ask me to review your code — I need to record your progress.
3. Before submission, rename the file. Replace `YOURNAME` with your full name (e.g., `CSI3680-Lab2-LoriXu.ipynb`)
4. Submit the **renamed and completed** `.ipynb` file to Moodle.
  - Make sure you submit your final edited `.ipynb` file — not a blank copy.

## Question 1 — Build a command-line study planner
### Question
How can a student use command-line options to customize a study checklist while passing the topics as positional arguments?

### Task
Complete `study_plan.sh`. It prints a study plan using these options:

| Option | Meaning | Default |
| --- | --- | --- |
| `-c COURSE` | Course name or code | `CSI3680` |
| `-d DAY` | Day or date label (use the supplied text as-is) | `today` |
| `-u` | Mark the plan urgent | `NORMAL` priority |
| `-h` | Print usage and exit successfully | — |

1. Use `getopts` with custom error handling to parse the options. Store option arguments in the supplied variables. Repeated settings use the last supplied value.
2. After parsing, use `OPTIND` and `shift` to leave only the study topics in the positional arguments. Options must precede topics in the tests.
3. Require at least one topic unless `-h` is encountered. Preserve quoted topics containing spaces as single items.
4. Print a heading and numbered checklist using the supplied output code. Return status `0` for a valid plan or help.
5. For an unknown option or missing option argument, print the specified error to standard error and exit `1`. If no topics remain, print `Error: Provide at least one study topic.` to standard error and exit `1`. Do not print a plan on error. `-h` prints usage and exits immediately when encountered.

The scaffold provides defaults, usage text, and checklist formatting. Complete the five TODOs for option parsing and validation.

### Expected output
For `bash study_plan.sh -c CSI3680 -d Friday -u "Bash loops" getopts`:
```text
Study plan: CSI3680 | Friday | URGENT
1. Review Bash loops
2. Review getopts
```
For `bash study_plan.sh "Shell variables"` (all defaults):
```text
Study plan: CSI3680 | today | NORMAL
1. Review Shell variables
```
For `bash study_plan.sh -h`:
```text
Usage: bash study_plan.sh [-c COURSE] [-d DAY] [-u] [-h] TOPIC [TOPIC ...]
```
Error cases:

| Command | Expected standard error | Exit status |
| --- | --- | --- |
| `bash study_plan.sh -d` | `Error: Option -d requires an argument.` | `1` |
| `bash study_plan.sh -x` | `Error: Unknown option -x.` | `1` |
| `bash study_plan.sh -u` | `Error: Provide at least one study topic.` | `1` |

### Your script

In [ ]:
%%writefile study_plan.sh
#!/bin/bash
# Print a customized study checklist from options and topic arguments.
todo() { printf 'Complete TODO %s before testing.\n' "$1" >&2; exit 1; }
course="CSI3680"
day="today"
priority="NORMAL"

usage() {
    echo 'Usage: bash study_plan.sh [-c COURSE] [-d DAY] [-u] [-h] TOPIC [TOPIC ...]'
}

# TODO 1: Fill in the option string. Use a leading colon for custom errors.
while getopts "" opt; do
    case "$opt" in
        # TODO 2: Assign the course, day, and priority for these options.
        c) todo 2 ;;
        d) todo 2 ;;
        u) todo 2 ;;
        h) usage; exit 0 ;;
        # TODO 3: Print the appropriate error to stderr and exit 1.
        :) todo 3 ;;
        \?) todo 3 ;;
    esac
done

# TODO 4: Remove the parsed options and their arguments using shift.
todo 4

# TODO 5: If no study topics remain, print the error to stderr and exit 1.
todo 5

# PROVIDED: Format the plan; "$@" preserves each quoted topic.
printf 'Study plan: %s | %s | %s\n' "$course" "$day" "$priority"
number=1
for topic in "$@"; do
    printf '%s. Review %s\n' "$number" "$topic"
    number=$((number + 1))
done
exit 0

In [ ]:
%%bash
bash study_plan.sh -c CSI3680 -d Friday -u "Bash loops" getopts
printf 'Exit status: %s\n\n' "$?"
bash study_plan.sh "Shell variables"
printf 'Exit status: %s\n\n' "$?"
bash study_plan.sh -h
printf 'Exit status: %s\n\n' "$?"
bash study_plan.sh -d
printf 'Exit status: %s\n\n' "$?"
bash study_plan.sh -x
printf 'Exit status: %s\n\n' "$?"
bash study_plan.sh -u
printf 'Exit status: %s\n' "$?"

### One more test with a different option order

In [ ]:
%%bash
bash study_plan.sh -u -d Monday -c "Script Programming" "Bash arrays"

### Your explanation
- What option string did you choose, and why? **Your answer**
- Why must you shift the arguments before checking for topics? **Your answer**
- Why does the supplied loop use `"$@"` rather than unquoted `$*`? **Your answer**

## Question 2 —  Back up and recover a course project
### Question
You are working on a course project over several days. How can you save a full copy first, back up only later changes, and recover the latest version using those backups?

### Task
Complete the five TODOs in `backup.sh`, then run the experiment below. Use `tar` with gzip.

The script backs up the fixed practice directory `lab2_project`:
```bash
bash backup.sh full lab2_backups/full.tgz
bash backup.sh incremental lab2_backups/day2.tgz
```

1. A **full backup** archives the entire `lab2_project` directory with `tar -czf`.
2. An **incremental backup** archives only regular files modified after the last successful backup. Refuse an incremental backup if the timestamp file does not exist.
3. After a successful backup, replace the saved timestamp with the backup's start-time marker. The marker and its cleanup are provided in the starter. If backup creation fails, keep the old timestamp.
4. If an incremental backup finds no changed files, print `No changed files; no archive created.` and leave the timestamp unchanged.
5. Print `Created full backup: FILE` or `Created incremental backup: FILE` after success. Return status `0` for success or no changes; return `1` for an error.
6. Run the full backup, two incremental backups, and restore test in order. Explain which archives are required to recover the latest project.

**Scope:** use the supplied test files and a new archive filename for each successful backup. Keep archives outside the source directory. Files are not edited while a backup runs. This exercise tracks added and modified regular files; deletion tracking, permissions recovery, and rotation are outside its scope.

### Command help
We already covered `tar -czf ARCHIVE DIRECTORY`. These additional pieces are provided for the incremental and recovery steps:

| Command | Meaning |
| --- | --- |
| `touch MARKER` | Set a marker file's modification time to the current time. |
| `find lab2_project -type f -newer MARKER` | Print regular files modified after the marker. |
| `tar -czf ARCHIVE -T LIST` | Create a gzip-compressed archive using paths from a text file, one path per line. |
| `tar -tzf ARCHIVE` | List archive contents without extracting them. |
| `tar -xzf ARCHIVE -C DESTINATION` | Extract into an existing destination directory. |
| `diff -r DIRECTORY1 DIRECTORY2` | Compare directory contents recursively; no output and status `0` mean they match. |

For this lab, filenames contain no embedded newlines. The given paths begin with `lab2_project/`, so the same relative paths are used in all three archives.

### Expected archive contents
Directory entries may also appear in the full archive. File-list order does not matter.

| Backup | Change before the backup | Regular files expected in the archive |
| --- | --- | --- |
| `full.tgz` | Initial project | `notes.txt`, `report.txt`, `data/scores.txt` |
| `day2.tgz` | Modify only `report.txt` | `report.txt` only |
| `day3.tgz` | Add only `summary.txt` | `summary.txt` only |

In particular, `report.txt` must **not** appear in `day3.tgz`: its changes were already saved by the previous backup.

### Sample data
Run this setup once before the experiment. Re-running it resets the named practice files and timestamp; then repeat the tests in order. Use a fresh notebook working directory for the first run.

In [ ]:
%%bash
set -e
mkdir -p lab2_project/data lab2_backups
printf 'Bash study notes\n' > lab2_project/notes.txt
printf 'Report draft\n' > lab2_project/report.txt
printf 'Alice,90\nBob,85\n' > lab2_project/data/scores.txt
# Reset only this exercise's known generated files.
rm -f lab2_project/summary.txt lab2_backups/.last_backup \
      lab2_backups/full.tgz lab2_backups/day2.tgz lab2_backups/day3.tgz \
      lab2_backups/empty.tgz lab2_backups/too_early.tgz

### Your script
Complete TODOs 1–5. Argument handling, temporary-file cleanup, and the empty-selection check are supplied.

In [ ]:
%%writefile backup.sh
#!/bin/bash
# Archive a course project; preserve the checkpoint if a backup fails.
todo() { printf 'Complete TODO %s before testing.\n' "$1" >&2; exit 1; }

# PROVIDED: Arguments, paths, and basic checks.
if [ "$#" -ne 2 ]; then
    echo 'Usage: bash backup.sh full|incremental ARCHIVE' >&2
    exit 1
fi
mode=$1
archive=$2
case "$mode" in
    full|incremental) ;;
    *) echo 'Error: Choose full or incremental.' >&2; exit 1 ;;
esac
source_dir='lab2_project'
backup_dir='lab2_backups'
tsfile="$backup_dir/.last_backup"
file_list="$backup_dir/.selected_files"
start_marker="$backup_dir/.backup_started"

if [ ! -d "$source_dir" ]; then
    echo 'Error: Source directory is missing.' >&2
    exit 1
fi
mkdir -p "$backup_dir" || exit 1
if [ ! -w "$backup_dir" ]; then
    echo 'Error: Backup directory is not writable.' >&2
    exit 1
fi
# Use the provided archive paths; run one backup at a time.
trap 'rm -f "$file_list" "$start_marker"' EXIT
touch "$start_marker" || exit 1
no_changes=0

create_archive() {
    if [ "$mode" = 'full' ]; then
        # TODO 1: Create a gzip-compressed archive of the entire source directory.
        todo 1
    else
        # TODO 2: If tsfile does not exist, print the message below to stderr
        # and return 1: Error: Run a full backup first.
        todo 2

        # TODO 3: Find regular files newer than tsfile and write their paths
        # to file_list. Use: find ... > "$file_list" || return 1
        todo 3

        # PROVIDED: Handle an empty selection without making an empty archive.
        if [ ! -s "$file_list" ]; then
            echo 'No changed files; no archive created.'
            no_changes=1
            return 0
        fi

        # TODO 4: Create the archive using tar with -T and file_list.
        todo 4
    fi
    # The status of the tar command is the function's return status.
}

# PROVIDED: Check success before advancing the timestamp.
if create_archive; then
    if [ "$no_changes" -eq 0 ]; then
        # TODO 5: Move start_marker to tsfile; exit 1 if mv fails.
        # This preserves the recorded start time. Use: mv ... || exit 1
        todo 5
        echo "Created $mode backup: $archive"
    fi
else
    echo 'Error: Backup failed; timestamp unchanged.' >&2
    exit 1
fi
exit 0

### Test 1 — Save the initial project
Run the completed script to create a full backup, then list the archive.

**Expected:** `Created full backup: lab2_backups/full.tgz`, the three initial files (and directory entries), and `Timestamp exists`.

In [ ]:
%%bash
set -e
bash backup.sh full lab2_backups/full.tgz
tar -tzf lab2_backups/full.tgz
[ -f lab2_backups/.last_backup ]
echo 'Timestamp exists'

### Test 2 — Save changes on two later days
The waits ensure that new modification times are later than the saved checkpoint. Run this cell once after Test 1.

**Expected output:**
```text
Created incremental backup: lab2_backups/day2.tgz
lab2_project/report.txt
Created incremental backup: lab2_backups/day3.tgz
lab2_project/summary.txt
```
The second incremental archive must not include the report again.

In [ ]:
%%bash
set -e
# Day 2: change one existing file.
sleep 1
printf 'Report revised on day 2\n' > lab2_project/report.txt
bash backup.sh incremental lab2_backups/day2.tgz
tar -tzf lab2_backups/day2.tgz

# Day 3: add one file; leave the report unchanged.
sleep 1
printf 'Project summary added on day 3\n' > lab2_project/summary.txt
bash backup.sh incremental lab2_backups/day3.tgz
tar -tzf lab2_backups/day3.tgz

### Test 3 — Recover the latest project
The provided test restores into a temporary folder. It leaves the working project intact. Read the commands and run the cell.

Extract the **full backup first**, followed by **day 2** and then **day 3**. Later archives overwrite earlier versions of the same file.

**Expected output:**
```text
Restored report:
Report revised on day 2
Restored summary:
Project summary added on day 3
PASS: Restored project matches the current project.
```

In [ ]:
%%bash
set -e
restore_dir=$(mktemp -d)
trap 'rm -rf "$restore_dir"' EXIT

tar -xzf lab2_backups/full.tgz -C "$restore_dir"
tar -xzf lab2_backups/day2.tgz -C "$restore_dir"
tar -xzf lab2_backups/day3.tgz -C "$restore_dir"

echo 'Restored report:'
cat "$restore_dir/lab2_project/report.txt"
echo 'Restored summary:'
cat "$restore_dir/lab2_project/summary.txt"
diff -r lab2_project "$restore_dir/lab2_project"
echo 'PASS: Restored project matches the current project.'

### Explain your results
1. Why does the full backup contain three files while `day2.tgz` contains only one? **Your answer**
2. Why is `report.txt` absent from `day3.tgz`? **Your answer**
3. Could you recover the entire latest project using only `full.tgz` and `day3.tgz`? Which version of `report.txt` would you get? **Your answer**
4. Why should a failed backup leave the saved timestamp unchanged? **Your answer**

### Test 4 — No changes
Run the next cell after Test 3. Expect `No changed files; no archive created.` and the PASS message. This checks that an empty selection is handled explicitly.

In [ ]:
%%bash
set -e
cp -p lab2_backups/.last_backup lab2_backups/.before_empty
bash backup.sh incremental lab2_backups/empty.tgz
[ ! -e lab2_backups/empty.tgz ]
[ ! lab2_backups/.last_backup -nt lab2_backups/.before_empty ]
[ ! lab2_backups/.before_empty -nt lab2_backups/.last_backup ]
rm -f lab2_backups/.before_empty
echo 'PASS: No archive created and timestamp unchanged.'

## Question 3 — Build and play Acey Deucey
### Question
Can you build a shell game that deals two visible cards and asks the player whether a hidden third card ranks strictly between them?

### Task
Complete the `aceydeucy.sh` starter using Bash functions, arrays, loops, `read`, conditions, and arithmetic. Deck creation, random selection, shuffling, card names, and input handling are provided. Read these helpers, then complete the five TODOs for ranks, ordering, redealing, predictions, and scoring. The functions follow the structure of the reference game:

| Function | Required behavior |
| --- | --- |
| `initializeDeck` | Fill a deck array with the 52 unique card IDs, `1` through `52`. |
| `pickCard` | Select an unused card at random, store its ID in `picked`, and mark it used. After 10 unsuccessful random attempts, scan for an unused card. |
| `shuffleDeck` | Call `pickCard` 52 times to build `newdeck`, with each card appearing once. |
| `showCard` | Convert a card ID into a readable `cardname`, such as `Queen of Hearts`. |
| `dealCards` | Take the first three shuffled cards. Order the first two by rank, display them, and keep the third hidden until the player answers. |
| `intro` | Explain the rules when any command-line argument is supplied. |

Use the following rules:

1. Each round starts with a fresh 52-card deck and a new shuffle. Within a round, a card ID must not repeat.
2. Suits are Hearts (IDs 1–13), Clubs (14–26), Spades (27–39), and Diamonds (40–52). Compute the suit group with `(card - 1) / 13 + 1`. Compute rank with `card % 13`, changing `0` to `13` for King. Ace is **low** (`1`); Jack is `11`, Queen is `12`, King is `13`.
3. Calculate `spread = higher_rank - lower_rank`. If the first two ranks are equal, print `No point in betting when they're the same rank!` and deal again without counting a game.
4. Prompt for `y`, `n`, or `q`. `y` predicts the third rank is strictly between the first two; `n` predicts it is not. `q` prints the score and exits without counting the current round. For invalid input, prompt again for the same cards without changing the score. If input ends, print the score and exit.
5. Reveal the third card after a valid `y` or `n` answer. A correct prediction is a win. Equality with either endpoint is **not between**: it loses for `y` and wins for `n`.
6. Track completed games and wins, starting at zero. Each valid `y` or `n` answer counts as one game. End with `You played G games and won W times.`

**Implementation hints:** `$((RANDOM % 52 + 1))` gives a candidate card ID. Store a selected card in `picked`; use `return` for success/failure status, not to pass back the card ID. Call `pickCard` directly so changes to the deck remain in the current shell.

### Expected output
This is **one possible round**, not a fixed random result. For visible ranks 6 and 8 and a third rank of 7, answering `y` wins:
```text
I've dealt:
6 of Hearts
8 of Clubs
The spread is 2. Do you think the next card will be between them? (y/n/q) y
I picked: 7 of Spades
You bet that it would be between the two, and it is. WIN!
```
The next round begins automatically. If the player then enters `q`, the final line is:
```text
You played 1 games and won 1 times.
```
Use these result messages:
```text
You bet that it would be between the two, and it is. WIN!
You bet that it would not be between the two, and it isn't. WIN!
Bad betting strategy. You lose.
```
For invalid input, display `Please enter y, n, or q.` and repeat the prompt.

### Your script

In [ ]:
%%writefile aceydeucy.sh
#!/bin/bash
# Acey Deucey: predict whether the third rank is strictly between two others.
todo() { printf 'Complete TODO %s before testing.\n' "$1" >&2; exit 1; }

# PROVIDED: Build the deck, pick unused cards, and shuffle.
initializeDeck() {
    local card
    deck=()
    for ((card=1; card<=52; card++)); do
        deck[$card]=$card
    done
}

pickCard() {
    local attempt randomcard
    for ((attempt=0; attempt<10; attempt++)); do
        randomcard=$((RANDOM % 52 + 1))
        if [ "${deck[$randomcard]}" -ne 0 ]; then
            picked=${deck[$randomcard]}
            deck[$randomcard]=0
            return 0
        fi
    done
    # Fallback: find an available card without repeatedly guessing.
    for ((randomcard=1; randomcard<=52; randomcard++)); do
        if [ "${deck[$randomcard]}" -ne 0 ]; then
            picked=${deck[$randomcard]}
            deck[$randomcard]=0
            return 0
        fi
    done
    return 1
}

shuffleDeck() {
    local count
    newdeck=()
    for ((count=1; count<=52; count++)); do
        pickCard || return 1
        newdeck[$count]=$picked
    done
}

# PROVIDED: Convert a card ID to its display name.
showCard() {
    local card=$1 suit rank
    suit=$(((card - 1) / 13 + 1))
    rank=$((card % 13))
    case "$suit" in
        1) suit='Hearts' ;;
        2) suit='Clubs' ;;
        3) suit='Spades' ;;
        4) suit='Diamonds' ;;
    esac
    case "$rank" in
        0) rank='King' ;;
        1) rank='Ace' ;;
        11) rank='Jack' ;;
        12) rank='Queen' ;;
    esac
    cardname="$rank of $suit"
}

# COMPLETE: Prepare and display a round.
dealCards() {
    card1=${newdeck[1]}
    card2=${newdeck[2]}
    card3=${newdeck[3]}

    # TODO 1: Set rank1, rank2, and rank3 from the card IDs using % 13.
    # Convert each zero rank to 13 so King is the highest rank.
    todo 1

    # TODO 2: If rank1 > rank2, swap rank1/rank2 AND card1/card2.
    # Hint: use a temporary variable so neither value is lost.
    todo 2

    showCard "$card1"; cardname1=$cardname
    showCard "$card2"; cardname2=$cardname
    showCard "$card3"; cardname3=$cardname
    printf "\nI've dealt:\n%s\n%s\n" "$cardname1" "$cardname2"
}

# PROVIDED: Instructions and final score.
intro() {
    echo 'Welcome to Acey Deucey! Ace is low; King is high.'
    echo 'Predict whether the third rank is strictly between the first two.'
    echo 'Enter y for between, n for not between, or q to quit.'
    echo 'Matching either endpoint counts as not between.'
}
printScore() {
    echo "You played $games games and won $won times."
}

games=0
won=0
if [ "$#" -gt 0 ]; then intro; fi

while true; do
    initializeDeck
    shuffleDeck || exit 1
    dealCards
    splitValue=$((rank2 - rank1))

    # TODO 3: If the spread is zero, print the equal-rank message and
    # continue the OUTER loop without asking for input or counting a game.
    todo 3

    # PROVIDED: Keep asking until the answer is valid; handle quit/EOF.
    while true; do
        printf 'The spread is %s. Do you think the next card will be between them? (y/n/q) ' "$splitValue"
        if ! read -r answer; then
            printf '\n'
            printScore
            exit 0
        fi
        case "$answer" in
            y|n) break ;;
            q) printScore; exit 0 ;;
            *) echo 'Please enter y, n, or q.' ;;
        esac
    done

    echo "I picked: $cardname3"

    # TODO 4: Set between=1 if rank3 is STRICTLY between rank1 and rank2;
    # otherwise set between=0. Use a Bash arithmetic condition.
    todo 4

    # TODO 5: Use between and answer to decide whether the prediction is
    # correct. Print the matching result message from the task.
    # Increase won only for a correct prediction; increase games for
    # every completed y/n round. The loop then deals another round.
    todo 5
done

### Try it
For interactive play, open a terminal in the directory containing your saved script and run:
```bash
bash aceydeucy.sh help
```
If using Colab, download the saved script to a machine with Bash to play interactively. You can still run the following piped tests in the notebook. Piped answers are not echoed as typed keyboard input.

The first test must finish with `You played 0 games and won 0 times.` The second must report an invalid answer, play exactly two rounds, and finish with `You played 2 games and won W times.`, where `W` can be `0`, `1`, or `2`.

In [ ]:
%%bash
printf 'q\n' | bash aceydeucy.sh
printf 'invalid\ny\nn\nq\n' | bash aceydeucy.sh